# Amazon ML Challenge 2026: Business Entity Resolution Pipeline
### End-to-End Execution Notebook for AWS SageMaker (`ml.m5.xlarge`)

This notebook runs the complete **Zero-RAM Disk-Indexed Multi-Stage Resolution Pipeline**:
1. **Zero-RAM SQLite B-Tree Indexing**: Streams 10.2M candidate records (Source 2 & Source 3) into an indexed SQLite disk database (`index.db`).
2. **Lexical Retrieval & Re-ranking**: Fast candidate blocking with frequency-filtered tokens and composite lexical bonus scoring.
3. **17-Dimensional Feature Extraction**: Fuzzy token ratios, Jaro-Winkler, containment indicators, and exact matches.
4. **LightGBM Classifier & Threshold Optimizer**: GBDT trained with leak-proof GroupKFold split, optimizing specifically for **Macro $F_{0.5}$**.
5. **Confusion Matrix & Metric Diagnostics**: Evaluates True/False Positives & Negatives on unseen validation entities.
6. **Submission Generation & Validation**: Produces competition-compliant `candidate_pairs.tsv` and `matching_results.tsv`.

--- 
## 1. Environment & Dependency Setup
Creates a Python virtual environment (`.venv`) and installs project dependencies.

In [ ]:
%%bash
cd ~/Amazon-ML-Hackathon-2026

# Create .venv if not already created
if [ ! -d ".venv" ]; then
    python3 -m venv .venv
    echo "Virtual environment .venv created."
fi

source .venv/bin/activate
pip install --upgrade pip
pip install pandas numpy lightgbm pyarrow scikit-learn boto3 matplotlib seaborn


--- 
## 2. Sync Latest Code from GitHub

In [ ]:
%%bash
if [ ! -d ".git" ] && [ -d "Amazon-ML-Hackathon-2026" ]; then
    cd Amazon-ML-Hackathon-2026
fi
git stash
git pull origin main


--- 
## 3. Verify Dataset Files
Ensures all 7 TSV dataset files are present in `dataset/train/` and `dataset/test/`.

In [ ]:
%%bash
cd ~/Amazon-ML-Hackathon-2026
echo "=== TRAIN DATASET ==="
ls -lh dataset/train/

echo ""
echo "=== TEST DATASET ==="
ls -lh dataset/test/


--- 
## 4. Run Training & Validation Pipeline (`--split train`)
Runs the pipeline on 50,000 Source 1 entities against the 10.2M candidate database.
- Generates `index.db` automatically if not present.
- Tunes the optimal probability threshold to maximize **Macro $F_{0.5}$**.
- Logs the full Confusion Matrix & Classification Report to console.
- Persists trained model to `output/lgb_model.pkl`.

In [ ]:
%%bash
cd ~/Amazon-ML-Hackathon-2026
source .venv/bin/activate

# Run training on 50,000 sample with top 30 candidates per entity
python3 code/main.py --sample-size 50000 --max-candidates 30 --split train --no-cache


--- 
## 5. Check Training Artifacts & Model Status

In [ ]:
import os
import glob
import pickle

print("=== Training Artifacts (output/train/) ===")
train_dir = "output/train"
if os.path.exists(train_dir):
    for f in sorted(os.listdir(train_dir)):
        p = os.path.join(train_dir, f)
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f"  - {f:35s}: {size_mb:8.2f} MB")
else:
    print("  (No output/train directory yet. Run Section 4 to train.)")

print("
=== Testing Outputs (output/test/) ===")
test_dir = "output/test"
if os.path.exists(test_dir):
    for f in sorted(os.listdir(test_dir)):
        p = os.path.join(test_dir, f)
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f"  - {f:35s}: {size_mb:8.2f} MB")
else:
    print("  (No output/test directory yet. Run Section 6 to run test inference.)")

model_path = "output/train/lgb_model.pkl" if os.path.exists("output/train/lgb_model.pkl") else "output/lgb_model.pkl"
if os.path.exists(model_path):
    with open(model_path, "rb") as f:
        m = pickle.load(f)
    print(f"
Trained Model Ready at {model_path}! Optimal Macro F0.5 Threshold: {m['optimal_threshold']:.3f}")


--- 
## 5b. Validation Confusion Matrix & Metrics Visualization
Loads cached validation pairs and evaluates True/False Positives/Negatives with an annotated heatmap.

In [ ]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Find latest cache file
cache_files = glob.glob("output/train/cache_features_train_*.pkl") + glob.glob("output/cache_features_train_*.pkl")
if not cache_files:
    print("No feature cache found. Please run Section 4 first.")
else:
    cache_file = sorted(cache_files)[-1]
    model_file = "output/train/lgb_model.pkl" if os.path.exists("output/train/lgb_model.pkl") else "output/lgb_model.pkl"
    
    print(f"Loading cached features from: {cache_file}")
    with open(cache_file, "rb") as f:
        cache = pickle.load(f)
    
    with open(model_file, "rb") as f:
        model_data = pickle.load(f)
    
    clf = model_data["clf"]
    optimal_thresh = model_data.get("optimal_threshold", 0.500)
    
    # Reconstruct 80/20 Grouped Validation split
    all_candidate_pairs = cache["all_candidate_pairs"]
    groups_arr = np.array([p[0] for p in all_candidate_pairs])
    unique_groups = pd.Series(groups_arr).unique()
    split_idx = int(len(unique_groups) * 0.8)
    val_groups = set(unique_groups[split_idx:])
    is_val = np.array([g in val_groups for g in groups_arr])
    
    X_val = pd.DataFrame(cache["features_list"])[is_val]
    y_val = np.array(cache["labels_list"])[is_val]
    val_groups_arr = groups_arr[is_val]
    
    # Generate predictions using optimal threshold
    val_probs = clf.predict_proba(X_val)[:, 1]
    val_preds = (val_probs >= optimal_thresh).astype(int)
    
    cm = confusion_matrix(y_val, val_preds)
    tn, fp, fn, tp = cm.ravel()
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pairwise_f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    pairwise_f05 = (1.25 * prec * rec) / (0.25 * prec + rec) if (0.25 * prec + rec) > 0 else 0.0
    
    # Entity-Level Macro F0.5 (Official Competition Metric)
    _, g_idx = np.unique(val_groups_arr, return_inverse=True)
    n_val_groups = len(np.unique(g_idx))
    tp_g = np.bincount(g_idx, weights=(val_preds == 1) & (y_val == 1), minlength=n_val_groups)
    fp_g = np.bincount(g_idx, weights=(val_preds == 1) & (y_val == 0), minlength=n_val_groups)
    fn_g = np.bincount(g_idx, weights=(val_preds == 0) & (y_val == 1), minlength=n_val_groups)
    p_den = tp_g + fp_g
    r_den = tp_g + fn_g
    prec_g = np.divide(tp_g, p_den, out=np.zeros_like(tp_g, dtype=float), where=p_den != 0)
    rec_g = np.divide(tp_g, r_den, out=np.zeros_like(tp_g, dtype=float), where=r_den != 0)
    f_den = 0.25 * prec_g + rec_g
    f05_g = np.divide(1.25 * prec_g * rec_g, f_den, out=np.zeros_like(prec_g), where=f_den != 0)
    f05_g[(tp_g == 0) & (fp_g == 0) & (fn_g == 0)] = 1.0
    competition_macro_f05 = float(np.mean(f05_g))
    
    print("=" * 60)
    print(f"   VALIDATION CONFUSION MATRIX (Threshold = {optimal_thresh:.3f})   ")
    print("=" * 60)
    print(f"                 PREDICTED NON-MATCH       PREDICTED MATCH")
    print(f"ACTUAL NON-MATCH   TN: {tn:10,d}           FP: {fp:10,d}")
    print(f"ACTUAL MATCH       FN: {fn:10,d}           TP: {tp:10,d}")
    print("-" * 60)
    print(f"  True Positives  (TP) : {tp:8,d}  (Correct matches)")
    print(f"  False Positives (FP) : {fp:8,d}  (False alarms)")
    print(f"  False Negatives (FN) : {fn:8,d}  (Missed true matches)")
    print(f"  True Negatives  (TN) : {tn:8,d}  (Correct non-matches)")
    print("-" * 60)
    print(f"  Pairwise Precision   : {prec * 100:.2f}%")
    print(f"  Pairwise Recall      : {rec * 100:.2f}%")
    print(f"  Pairwise F1-Score    : {pairwise_f1:.4f}")
    print(f"  Pairwise F0.5        : {pairwise_f05:.4f}")
    print(f"  Macro F0.5 (Metric)  : {competition_macro_f05:.4f}  [OFFICIAL LEADERBOARD]")
    print("=" * 60)
    
    # Plot heatmap
    plt.figure(figsize=(7, 5))
    annot = np.array([[f"{tn:,}
(TN)", f"{fp:,}
(FP)"], [f"{fn:,}
(FN)", f"{tp:,}
(TP)"]])
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=True,
                xticklabels=["Predicted Non-Match", "Predicted Match"],
                yticklabels=["Actual Non-Match", "Actual Match"])
    plt.title(f"Validation Confusion Matrix (Threshold = {optimal_thresh:.3f})", fontsize=13, pad=12)
    plt.ylabel("Ground Truth Label", fontsize=11)
    plt.xlabel("Model Prediction", fontsize=11)
    plt.tight_layout()
    plt.show()


--- 
## 6. Run Test Split Inference (`--split test`)
Executes candidate retrieval and LightGBM scoring on `test_source1.tsv`.
Produces final competition outputs:
- `output/candidate_pairs.tsv`
- `output/matching_results.tsv`

In [ ]:
%%bash
cd ~/Amazon-ML-Hackathon-2026
source .venv/bin/activate

# Run inference on the full test set
python3 code/main.py --split test --max-candidates 30


--- 
## 7. Official Submission Format Validation

In [ ]:
%%bash
cd ~/Amazon-ML-Hackathon-2026
source .venv/bin/activate

# Check if official validate_submission.py exists
VAL_SCRIPT=$(find . -name "validate_submission.py" | head -n 1)
if [ -f "$VAL_SCRIPT" ]; then
    python3 "$VAL_SCRIPT" \
        --matching output/test/matching_results.tsv \
        --candidate output/test/candidate_pairs.tsv \
        --test-dir dataset/test/
else
    echo "Validation script not found. Checking basic format in Python..."
fi


--- 
## 8. Preview Final Submission Files

In [ ]:
import pandas as pd

cand_file = "output/test/candidate_pairs.tsv" if os.path.exists("output/test/candidate_pairs.tsv") else "output/candidate_pairs.tsv"
    df_cand = pd.read_csv(cand_file, sep="\t", nrows=10)
print("=== Candidate Pairs (Top 10 Rows) ===")
print(df_cand)

print("\n=== Matching Results (Top 10 Rows) ===")
match_file = "output/test/matching_results.tsv" if os.path.exists("output/test/matching_results.tsv") else "output/matching_results.tsv"
    df_match = pd.read_csv(match_file, sep="\t", nrows=10)
print(df_match)


--- 
## 9. (Optional) Sync Output Files to S3 Bucket

In [ ]:
%%bash
# Sync final outputs to your S3 bucket
aws s3 cp output/test/matching_results.tsv s3://sagemaker-eu-north-1-583664563299/amazon-ml-challenge-2026/output/matching_results.tsv
aws s3 cp output/test/candidate_pairs.tsv s3://sagemaker-eu-north-1-583664563299/amazon-ml-challenge-2026/output/candidate_pairs.tsv
echo "Outputs backed up to S3 bucket!"
